# Preparación del dataset para la U-Net (2D y 2.5D)

**Objetivo.** Convertir los volúmenes ya limpios del notebook 01 y las máscaras de la articulación en el dataset con el que se entrenarán y evaluarán la **U-Net 2D** y la **U-Net 2.5D**: imágenes normalizadas, del tamaño que la red necesita, emparejadas con su máscara, con particiones **por paciente** y con el aumento de datos definido.

**Dónde encaja.**

```
.raw → [01: preparación de imágenes] → MedSAM2 → corrección manual → [02: ESTE NOTEBOOK] → [03: entrenamiento] → app
```

El notebook 01 preparó la *imagen* (N4, recorte de percentiles, `uint8`) y es común a MedSAM2 y a la U-Net. Este notebook prepara el *dataset de entrenamiento*, que solo tiene sentido con máscaras.

**Entradas.**
- `data/processed/SIJ_<n>.npz` (salida del notebook 01): `imgs` `(D, 400, 400)` `uint8`.
- `data/masks/SIJ_<n>.npz` (clave `mask`) o `SIJ_<n>.nii.gz`: máscara binaria `(D, 400, 400)`, 0 = fondo, 1 = articulación. **Este formato es una propuesta que el equipo debe confirmar con quien genere las máscaras de MedSAM2.**

**Mientras no existan las máscaras reales**, el notebook genera **máscaras provisionales sintéticas** (dos franjas que *no son anatomía*) para que todo el flujo se pueda ejecutar y probar. Se marcan en el archivo de parámetros con `"mask_source": "provisional"`, y el notebook avisa en cada paso.

**Salidas** (en `data/unet/`, ignorada por git):
- `SIJ_<n>.npz` con `image` `(D, 416, 416)` `float32` normalizada, `mask` `(D, 416, 416)` `uint8` y `body` (máscara del cuerpo).
- `splits.json`: las particiones por paciente.
- `unet_prep_params.json`: todas las decisiones y estadísticas, para trazabilidad.

**Índice**
1. Parámetros.
2. Carga de los volúmenes procesados.
3. Máscaras: formato, carga y verificación.
4. Normalización de intensidad (z-score por volumen).
5. Tamaño de entrada: relleno a 416×416.
6. Inventario de cortes y desbalance de clases.
7. Entrada 2D frente a 2.5D.
8. Particiones por paciente.
9. Aumento de datos.
10. El dataset por cortes (lo que consumirá el entrenamiento).
11. Guardado y verificación.
12. Resumen, decisiones abiertas y pasos siguientes.

**Cómo ejecutar.** Igual que el notebook 01 (`uv sync` y kernel de `.venv`). Requiere haber ejecutado antes el notebook 01, que crea `data/processed/`.

**Advertencia:** como en el notebook 01, las salidas contienen imágenes de pacientes y no deben subirse al repositorio (`uv run nbstripout --install`).

**Dónde está el código.** Las funciones están en `src/sij_prep/unet_data.py` y no en el notebook, porque el entrenamiento (notebook 03) y la **aplicación** deben preparar las imágenes **exactamente igual**. Si la app normalizara distinto que el entrenamiento, el modelo recibiría datos que nunca vio.

## 1. Parámetros

**Qué se hace.** Se fijan en una sola celda las rutas, el tamaño de entrada, el modo de la red y la semilla.

**Por qué.** Igual que en el notebook 01: cambiar una decisión (por ejemplo, el tamaño) debe ser editar una línea y volver a ejecutar, no buscar números repartidos por el notebook. La **semilla** fija hace que el aumento de datos y las figuras salgan iguales en cada ejecución, de modo que los resultados se pueden reproducir y comparar.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from sij_prep.io import load_npz, repo_root, save_params_json
from sij_prep.unet_data import (
    TARGET_SIZE, AugmentConfig, SliceDataset, augment, check_splits, find_mask_file,
    load_mask, load_unet_volumes, lopo_splits, make_placeholder_mask, pad_to_multiple,
    prepare_volume, slice_input, unpad,
)

ROOT = repo_root()
PROCESSED_DIR = ROOT / "data" / "processed"   # salida del notebook 01
MASK_DIR = ROOT / "data" / "masks"            # máscaras corregidas (MedSAM2 + revisión manual)
OUT_DIR = ROOT / "data" / "unet"              # salida de este notebook

SEED = 42
rng = np.random.default_rng(SEED)

if not PROCESSED_DIR.is_dir() or not list(PROCESSED_DIR.glob("SIJ_*.npz")):
    raise FileNotFoundError(f"No hay volúmenes en {PROCESSED_DIR}. Ejecute primero el notebook 01.")

print("Tamaño de entrada:", TARGET_SIZE, "| máscaras en:", MASK_DIR, "| salida:", OUT_DIR)

## 2. Carga de los volúmenes procesados

**Qué se hace.** Se leen los 9 `SIJ_<n>.npz` de `data/processed/` en orden numérico y se comprueba que sean `uint8` con forma `(D, 400, 400)`.

**Por qué.** Este notebook confía en lo que produjo el 01; si alguien cambia el 01 (otro tamaño, otro tipo de dato), aquí debe fallar de inmediato con un mensaje claro y no entrenar en silencio con datos distintos. El orden numérico (`SIJ_2` antes que `SIJ_10`) evita que las particiones cambien según cómo el sistema ordene los nombres.

**Qué mirar.** 9 filas, `uint8`, 400×400 y entre 18 y 24 cortes. En total son unos 180 cortes: es un dataset **muy pequeño**, y eso condiciona casi todas las decisiones que siguen.

In [ ]:
files = sorted(PROCESSED_DIR.glob("SIJ_*.npz"), key=lambda p: int(p.stem.split("_")[1]))
vols_u8 = {}
for f in files:
    imgs, spacing = load_npz(f)
    assert imgs.dtype == np.uint8 and imgs.ndim == 3 and imgs.shape[1:] == (400, 400), (
        f"{f.name}: se esperaba uint8 (D, 400, 400) y llegó {imgs.dtype} {imgs.shape}")
    vols_u8[f.stem] = imgs
PATIENTS = list(vols_u8)

print(f"{'volumen':8s} {'forma':>16s}")
for p, v in vols_u8.items():
    print(f"{p:8s} {str(v.shape):>16s}")
print(f"Total: {len(PATIENTS)} pacientes, {sum(v.shape[0] for v in vols_u8.values())} cortes")

## 3. Máscaras: formato, carga y verificación

**Formato propuesto** (a confirmar con quien haga MedSAM2):
- Un archivo por paciente con el mismo nombre que la imagen: `data/masks/SIJ_<n>.npz` (clave `mask`) o `SIJ_<n>.nii.gz`.
- Misma forma que la imagen procesada, `(D, 400, 400)`, en el mismo orden de cortes, sin rotar ni voltear.
- Binaria: 0 = fondo, 1 = articulación. Se aceptan también {0, 255}.
- **Todos los cortes** del volumen, incluidos los que no muestran la articulación (quedan en cero).

**Por qué cada punto.**
- *Misma forma y orden:* la máscara se empareja con la imagen píxel a píxel. Un corrimiento de un solo corte haría que la red aprendiera a dibujar la articulación sobre la anatomía equivocada, y nada lo detectaría salvo mirar las figuras.
- *Binaria:* la propuesta segmenta "la articulación", una sola clase. Si el equipo decide separar izquierda y derecha, habría que cambiar `load_mask`, la función de pérdida y el aumento (la inversión horizontal intercambiaría las etiquetas). Por eso `load_mask` falla si encuentra otros valores, en vez de adivinar.
- *Todos los cortes:* la red debe aprender también a **no** dibujar nada en los cortes sin articulación. Si solo se le dan cortes positivos, en un paciente nuevo marcará algo en todos los cortes (falsos positivos).

**Máscaras provisionales.** Si un paciente no tiene máscara en `data/masks/`, se genera una sintética con `make_placeholder_mask` y el notebook lo dice. Sirven **solo para probar el código**: cualquier número calculado con ellas (desbalance, cobertura) es ficticio.

**Qué mirar.** La tabla dice de dónde salió cada máscara. En la figura, el contorno debe caer sobre las articulaciones sacroilíacas; con máscaras provisionales caerá en un lugar aproximado, y eso es lo esperado.

In [ ]:
masks, mask_source = {}, {}
for p, v in vols_u8.items():
    mf = find_mask_file(MASK_DIR, p) if MASK_DIR.is_dir() else None
    if mf is not None:
        m = load_mask(mf)
        mask_source[p] = f"real ({mf.name})"
    else:
        m = make_placeholder_mask(v.shape)
        mask_source[p] = "PROVISIONAL (sintética)"
    assert m.shape == v.shape, f"{p}: máscara {m.shape} distinta de la imagen {v.shape}"
    masks[p] = m

USING_PLACEHOLDER = any(s.startswith("PROVISIONAL") for s in mask_source.values())
for p in PATIENTS:
    print(f"{p:8s} {mask_source[p]}")
if USING_PLACEHOLDER:
    print("\n⚠️  Hay máscaras PROVISIONALES: sirven para probar el código, NO para entrenar ni reportar resultados.")

In [ ]:
def overlay_grid(vol, msk, title, n=6):
    # n cortes repartidos entre el primero y el último, con el contorno de la máscara.
    ks = np.linspace(0, vol.shape[0] - 1, n).round().astype(int)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.3), layout="constrained")
    for ax, k in zip(axes, ks):
        ax.imshow(vol[k], cmap="gray")
        if msk[k].any():
            ax.contour(msk[k], levels=[0.5], colors="#E69F00", linewidths=1.2)
        ax.set_title(f"corte {k}" + ("" if msk[k].any() else " (sin máscara)"), fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.show()

overlay_grid(vols_u8["SIJ_1"], masks["SIJ_1"], f"SIJ_1 — máscara {mask_source['SIJ_1']}")

## 4. Normalización de intensidad: z-score por volumen

**Qué se hace.** Para cada volumen se calcula la máscara del cuerpo (la misma función `body_mask` del notebook 01) y se transforma la imagen a $z = (x - \mu)/\sigma$, con la media $\mu$ y la desviación $\sigma$ **de los vóxeles dentro del cuerpo**. Se guarda en `float32`.

**Por qué no se usa directamente el `uint8` 0–255 del notebook 01.** Ese formato es el que exige MedSAM2. Para la U-Net hay dos razones para normalizar otra vez:
1. **Las redes entrenan mejor con entradas centradas en 0 y con escala ~1.** Con valores 0–255 los primeros gradientes son enormes y el entrenamiento es inestable. Es el estándar de nnU-Net para RM (Isensee et al., 2021).
2. **Los volúmenes siguen sin ser comparables.** El notebook 01 mostró que, después de su normalización, las medianas van de ~70 a ~100 según el paciente. El z-score lleva cada volumen a media 0 y desviación 1 dentro del cuerpo, así que el mismo tejido queda en rangos parecidos entre pacientes.

**Por qué dentro del cuerpo.** El aire de fondo ocupa entre el 8 % y el 26 % del volumen según el paciente. Si se incluyera, la media y la desviación dependerían de cuánto aire hay alrededor y no del tejido.

**Por qué por volumen y no por corte.** La U-Net 2.5D compara cada corte con sus vecinos. Si cada corte se normalizara por separado, el mismo tejido cambiaría de valor de un corte al siguiente y la red vería diferencias que no existen. Es el mismo argumento que se usó para MedSAM2 en el notebook 01.

**Por qué no se recorta de nuevo.** El notebook 01 ya recortó los percentiles 0.5–99.5, así que no quedan valores extremos que el z-score tenga que proteger.

**Qué mirar.** En los diagramas de cajas, antes las cajas están a alturas distintas según el paciente; después, todas centradas en ~0 y con anchos parecidos. No serán idénticas: el z-score iguala la media y la desviación, no la forma de la distribución.

In [ ]:
prepared, norm_stats = {}, {}
for p, v in vols_u8.items():
    z, body, st = prepare_volume(v, TARGET_SIZE)
    m_pad, _ = pad_to_multiple(masks[p], TARGET_SIZE, value=0)
    prepared[p] = (z, m_pad.astype(np.uint8), body)
    norm_stats[p] = st

print(f"{'volumen':8s} {'media cuerpo':>12s} {'desv. cuerpo':>12s} {'valor del fondo':>16s}")
for p, st in norm_stats.items():
    print(f"{p:8s} {st['mean']:12.1f} {st['std']:12.1f} {st['background_value']:16.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
sub = lambda a: a[rng.random(a.shape) < 0.05]   # submuestra del 5 % para graficar rápido
a1.boxplot([sub(vols_u8[p][unpad(prepared[p][2], norm_stats[p]["pad"]).astype(bool)]) for p in PATIENTS],
           tick_labels=PATIENTS, showfliers=False)
a1.set_title("Antes: uint8 del notebook 01 (dentro del cuerpo)"); a1.set_ylabel("intensidad")
a2.boxplot([sub(prepared[p][0][prepared[p][2].astype(bool)]) for p in PATIENTS],
           tick_labels=PATIENTS, showfliers=False)
a2.axhline(0, color="#999", lw=0.8); a2.set_title("Después: z-score por volumen"); a2.set_ylabel("z")
for a in (a1, a2): a.tick_params(axis="x", rotation=45)
plt.show()

## 5. Tamaño de entrada: relleno de 400×400 a 416×416

**Qué se hace.** Se añaden 8 píxeles en cada borde (arriba, abajo, izquierda y derecha) con el valor que tiene el aire tras la normalización. A la máscara se le añaden ceros.

**Por qué hay que cambiar el tamaño.** La U-Net reduce la imagen a la mitad en cada nivel del codificador y luego la vuelve a ampliar, uniendo cada nivel con su par (las *skip connections*). Con 5 niveles, como en los codificadores tipo ResNet, el lado debe ser múltiplo de $2^5 = 32$. Como 400/32 = 12.5, en algún nivel aparecería un lado impar y los tamaños dejarían de coincidir al unir los niveles.

**Por qué rellenar y no las otras opciones.**

| Opción | Problema |
|---|---|
| Recortar a 384×384 | El cuerpo llega hasta el borde de la imagen en los 9 volúmenes (se comprobó con la máscara del cuerpo): se perdería tejido. |
| Redimensionar a 256×256 | Los píxeles pasarían de 0.6 mm a ~0.94 mm. La articulación es una estructura delgada (pocos milímetros) y perdería detalle justo en el borde que queremos segmentar. Además, la máscara redimensionada dejaría de ser exacta. |
| **Rellenar a 416×416** | No se pierde ni se interpola nada; la máscara sigue siendo exacta y la operación se deshace exactamente (`unpad`). Cuesta un 8 % más de píxeles. |

**Por qué con el valor del aire.** Si se rellenara con 0 después del z-score, el borde nuevo tendría el valor de un tejido *promedio* (z = 0) y se vería como una franja gris que no existe en ninguna imagen real.

**Para la app.** La app aplica `prepare_volume`, predice sobre 416×416 y devuelve la máscara a 400×400 con `unpad`.

**Qué mirar.** La imagen rellenada debe verse igual que la original, con un marco fino del color del aire. La verificación debe imprimir `True`.

In [ ]:
z1, m1, _ = prepared["SIJ_1"]
k = z1.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(9, 4.4), layout="constrained")
axes[0].imshow(vols_u8["SIJ_1"][k], cmap="gray"); axes[0].set_title("Original 400×400 (uint8)")
axes[1].imshow(z1[k], cmap="gray"); axes[1].set_title("Normalizada y rellenada 416×416 (z)")
axes[1].add_patch(plt.Rectangle((7.5, 7.5), 400, 400, fill=False, ec="#E69F00", lw=1, ls="--"))
for a in axes: a.set_xlabel("columna (píxel)"); a.set_ylabel("fila (píxel)")
plt.show()

roundtrip = all(np.array_equal(unpad(prepared[p][1], norm_stats[p]["pad"]), masks[p]) for p in PATIENTS)
print("unpad(relleno(máscara)) == máscara original en los 9 volúmenes:", roundtrip)
assert roundtrip

## 6. Inventario de cortes y desbalance de clases

**Qué se hace.** Para cada corte se calcula la fracción de píxeles que son articulación. Se resume cuántos cortes contienen articulación y qué fracción de los píxeles es primer plano.

**Por qué importa.** Ambos números deciden cómo entrenar:
- **Desbalance de píxeles.** Si la articulación ocupa, por ejemplo, el 1 % de los píxeles, una red que prediga "todo fondo" acierta el 99 % de los píxeles y aprende muy poco. Por eso en el entrenamiento se usará **Dice + entropía cruzada binaria** (BCE). El Dice solo mira el solapamiento con la articulación y no se deja engañar por el fondo; la BCE da gradientes estables al principio.
- **Cortes sin articulación.** Son necesarios (sección 3), pero si son muchos la red los verá más que los positivos. Si el entrenamiento tiende a predecir máscaras vacías, se pueden **sobremuestrear los cortes positivos** (`SliceDataset.positive_flags()` sirve para eso). Es una decisión del notebook de entrenamiento, no de este.

**Qué mirar.** El gráfico muestra, para cada paciente, la fracción de articulación por corte, de anterior a posterior. Con máscaras reales se espera cero en los primeros y últimos cortes y una meseta en los centrales. **Con máscaras provisionales estos números no significan nada.**

In [ ]:
inventory = {}
fig, ax = plt.subplots(figsize=(10, 3.6), layout="constrained")
for p in PATIENTS:
    m = masks[p]
    frac = m.reshape(m.shape[0], -1).mean(axis=1)
    inventory[p] = {"n_slices": int(m.shape[0]), "n_positive": int((frac > 0).sum()),
                    "fg_fraction": float(m.mean()), "per_slice_fraction": [round(float(f), 6) for f in frac]}
    ax.plot(np.arange(m.shape[0]) / (m.shape[0] - 1), 100 * frac, marker=".", lw=1, label=p)
ax.set_xlabel("posición relativa del corte (0 = anterior, 1 = posterior)")
ax.set_ylabel("% de píxeles de articulación")
ax.set_title("Fracción de articulación por corte" + ("  [MÁSCARAS PROVISIONALES]" if USING_PLACEHOLDER else ""))
ax.legend(ncol=9, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.22))
plt.show()

n_sl = sum(v["n_slices"] for v in inventory.values())
n_pos = sum(v["n_positive"] for v in inventory.values())
fg = np.mean([masks[p].mean() for p in PATIENTS])
print(f"Cortes con articulación: {n_pos} de {n_sl} ({100 * n_pos / n_sl:.0f} %)")
print(f"Píxeles de articulación (promedio por volumen): {100 * fg:.2f} %")

## 7. Entrada 2D frente a 2.5D

**Qué se hace.** Para cada corte `k` se construye la entrada de la red:
- **2D:** un solo canal, el corte `k`, de forma `(1, 416, 416)`.
- **2.5D:** tres canales, los cortes `k-1`, `k` y `k+1`, de forma `(3, 416, 416)`. La red predice la máscara **solo del corte central**.

**Por qué 2.5D.** Mirar un solo corte puede ser ambiguo: una línea oscura puede ser la articulación o el borde de otra estructura. Los vecinos dan contexto 3D (la articulación continúa entre cortes) sin el costo de una U-Net 3D. Con solo 18 a 24 cortes por volumen, una red 3D tendría muchos más parámetros y muy pocos ejemplos para aprenderlos.

**Por qué solo ±1 corte.** Los cortes están a 3 mm (anisotropía 5:1, notebook 01): con ±1 la pila cubre 9 mm. Con ±2 serían 15 mm y los cortes extremos mostrarían anatomía bastante distinta. Además, 3 canales coinciden con lo que espera un **codificador preentrenado en ImageNet** (RGB), que se puede usar sin modificarlo. A diferencia de MedSAM2, aquí los 3 canales **sí** traen información distinta.

**Por qué se repite el corte en los extremos.** El primer corte no tiene vecino anterior; se usa `[0, 0, 1]`. Si se rellenara con un corte negro, la red vería un canal artificial que no aparece en ningún otro corte y podría aprender a reaccionar a él.

**Comparación justa.** Para que la diferencia 2D/2.5D se pueda atribuir al contexto, ambas redes deben usar la **misma** arquitectura, normalización, particiones, aumento y número de épocas. Solo cambia el número de canales de entrada.

**Qué mirar.** Los tres canales deben verse parecidos pero no idénticos (la anatomía cambia un poco cada 3 mm), y la máscara corresponde al canal central. En el corte 0, los dos primeros canales son idénticos.

In [ ]:
z1, m1, _ = prepared["SIJ_1"]
fig, axes = plt.subplots(2, 4, figsize=(13, 6.6), layout="constrained")
for row, k in enumerate([z1.shape[0] // 2, 0]):
    x = slice_input(z1, k, "2.5d")
    names = [f"canal 0: corte {max(k - 1, 0)}", f"canal 1: corte {k} (central)", f"canal 2: corte {min(k + 1, z1.shape[0] - 1)}"]
    for c in range(3):
        axes[row, c].imshow(x[c], cmap="gray"); axes[row, c].set_title(names[c], fontsize=9)
    axes[row, 3].imshow(x[1], cmap="gray")
    if m1[k].any():
        axes[row, 3].contour(m1[k], levels=[0.5], colors="#E69F00", linewidths=1.2)
    axes[row, 3].set_title(f"objetivo: máscara del corte {k}", fontsize=9)
for a in axes.ravel(): a.axis("off")
fig.suptitle("Entrada 2.5D de SIJ_1: corte central (arriba) y primer corte (abajo, borde repetido)")
plt.show()
print("Forma 2D:", slice_input(z1, 5, "2d").shape, "| forma 2.5D:", slice_input(z1, 5, "2.5d").shape)

## 8. Particiones por paciente

**Qué se hace.** Validación cruzada **dejando un paciente fuera** (*leave-one-patient-out*, LOPO) con 9 particiones. En la partición *i*:
- **prueba:** el paciente *i*, que solo se usa al final para medir;
- **validación:** el paciente *i+1* (de forma circular), para elegir la mejor época (*early stopping*) y ajustar hiperparámetros;
- **entrenamiento:** los 7 restantes.

**Por qué por paciente.** Los cortes de un mismo paciente se parecen mucho entre sí (misma anatomía y mismo contraste). Si los cortes de un paciente se repartieran entre entrenamiento y prueba, la red sería evaluada en algo casi visto y el Dice saldría inflado. La propuesta lo exige explícitamente.

**Por qué LOPO.** Con 9 pacientes, separar un 20 % de prueba dejaría 1 o 2 pacientes, y el resultado dependería de cuáles tocaron. Con LOPO **cada paciente se evalúa una vez** y se obtienen 9 valores de Dice. Su media es el desempeño y su desviación es la **estabilidad** que pide el objetivo específico 4.

**Por qué un paciente de validación separado.** Si la mejor época se eligiera mirando al paciente de prueba, la prueba dejaría de ser independiente (optimismo por selección). El paciente de validación cumple ese papel y el de prueba no se toca hasta el final.

**Costo.** 9 particiones × 2 modelos = 18 entrenamientos. Si en Colab resulta demasiado, la alternativa es 3 particiones de 3 pacientes (6 entrenamientos), a cambio de menos datos de entrenamiento por partición. Se puede cambiar aquí sin tocar el resto.

**Qué mirar.** La tabla y la verificación automática: ningún paciente en dos conjuntos de la misma partición y cada paciente como prueba exactamente una vez.

In [ ]:
folds = lopo_splits(PATIENTS)
check_splits(folds, PATIENTS)
print(f"{'partición':>9s}  {'prueba':7s} {'valid.':7s} entrenamiento")
for f in folds:
    print(f"{f['fold']:>9d}  {f['test'][0]:7s} {f['val'][0]:7s} {', '.join(f['train'])}")
print("\nVerificación: sin fuga entre conjuntos y cada paciente es prueba una vez ✓")

## 9. Aumento de datos

**Qué se hace.** Durante el entrenamiento, cada vez que la red pide un corte se le aplica una transformación aleatoria distinta. El aumento **no se guarda en disco**: se genera al vuelo, así cada época ve variaciones nuevas. Solo se aplica en entrenamiento, nunca en validación ni en prueba.

**Por qué.** Con ~140 cortes de entrenamiento por partición, la red puede memorizarlos (sobreajuste). El aumento le muestra variaciones *plausibles* de la misma anatomía. La regla para elegir cada transformación es: **solo variaciones que podrían aparecer en una RM real de este protocolo.**

| Transformación | Rango | Por qué |
|---|---|---|
| Inversión horizontal | 50 % | La pelvis es aproximadamente simétrica izquierda-derecha y la máscara es de una sola clase. *Si se separan las clases izquierda/derecha, hay que desactivarla o intercambiar las etiquetas.* |
| Rotación | ±10° | Pequeñas diferencias en la posición del paciente y en la angulación del plano. Rotaciones grandes no ocurren: el protocolo fija el plano coronal oblicuo. |
| Escala | 0.9–1.1 | Diferencias de tamaño corporal entre pacientes. |
| Traslación | ±20 px (12 mm) | El paciente no queda siempre en la misma posición dentro del campo de visión. |
| Gamma, contraste y brillo | γ 0.8–1.25, ×0.9–1.1, ±0.1 | Variación de contraste entre exámenes y equipos que el z-score no elimina del todo. |
| Ruido gaussiano | σ = 0.05 | Diferencias de relación señal/ruido. |

**Qué se excluye y por qué.**
- *Inversión vertical:* una pelvis cabeza abajo no existe en estos estudios.
- *Deformaciones fuertes:* cambian la forma de la articulación, que es justo lo que queremos aprender. Una deformación elástica suave es una mejora opcional.

**Detalles que evitan errores silenciosos.**
- La **misma** transformación geométrica se aplica a los 3 canales y a la máscara. Si no, en 2.5D los cortes vecinos dejarían de estar alineados, y la máscara dejaría de coincidir con la imagen.
- La imagen se interpola de forma lineal y la máscara por vecino más cercano, para que siga siendo 0/1.
- Lo que entra por los bordes al rotar se rellena con el valor del aire.

**Qué mirar.** Seis versiones aumentadas del mismo corte: deben verse como RM plausibles, con el contorno de la máscara siempre pegado a la misma anatomía.

In [ ]:
z1, m1, _ = prepared["SIJ_1"]
k = z1.shape[0] // 2
x0, y0 = slice_input(z1, k, "2.5d"), m1[k]
aug_rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 7, figsize=(19, 3.2), layout="constrained")
axes[0].imshow(x0[1], cmap="gray", vmin=-3, vmax=4); axes[0].set_title("original", fontsize=9)
axes[0].contour(y0, levels=[0.5], colors="#E69F00", linewidths=1)
for a in axes[1:]:
    xa, ya = augment(x0, y0, aug_rng, AugmentConfig())
    a.imshow(xa[1], cmap="gray", vmin=-3, vmax=4)
    if ya.any():
        a.contour(ya, levels=[0.5], colors="#E69F00", linewidths=1)
    a.set_title("aumentada", fontsize=9)
for a in axes: a.axis("off")
plt.show()

# Control: con todos los rangos en cero, el aumento debe devolver exactamente la entrada.
ident = AugmentConfig(p_hflip=0, max_rotation_deg=0, scale_range=(1, 1), max_shift_px=0,
                      gamma_range=(1, 1), contrast_range=(1, 1), brightness_shift=0, noise_std=0)
xi, yi = augment(x0, y0, aug_rng, ident)
assert np.allclose(xi, x0, atol=1e-4) and np.array_equal(yi, y0)
print("Control de identidad: sin transformaciones, la salida es igual a la entrada ✓")

## 10. El dataset por cortes

**Qué se hace.** `SliceDataset` reúne todo lo anterior: dado un conjunto de pacientes (los de entrenamiento, validación o prueba de una partición) y un modo (`"2d"` o `"2.5d"`), devuelve por cada corte el par `(x, y)`, con `x` de forma `(C, 416, 416)` e `y` de forma `(1, 416, 416)`, y aplica el aumento si se le pide.

**Por qué así.**
- **Se guardan volúmenes, no cortes ni pilas.** Las pilas 2.5D se arman al vuelo; guardarlas triplicaría el disco y crearía dos fuentes de verdad que pueden desincronizarse.
- **No depende de PyTorch.** Tiene `__len__` y `__getitem__`, que es todo lo que necesita `torch.utils.data.DataLoader`. Así este notebook corre sin instalar PyTorch, y el de entrenamiento lo usa directamente:

```python
from torch.utils.data import DataLoader
vols = load_unet_volumes(OUT_DIR)
fold = folds[0]
train_ds = SliceDataset(vols, fold["train"], mode="2.5d", augment=True, seed=SEED)
val_ds   = SliceDataset(vols, fold["val"],   mode="2.5d", augment=False)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
```

**Qué mirar.** Para la partición 0, el número de cortes por conjunto y las formas de `x` e `y` en 2D y 2.5D.

In [ ]:
vols_mem = {p: (prepared[p][0], prepared[p][1]) for p in PATIENTS}
f0 = folds[0]
for mode in ("2d", "2.5d"):
    ds = {s: SliceDataset(vols_mem, f0[s], mode=mode, augment=(s == "train"), seed=SEED) for s in ("train", "val", "test")}
    x, y = ds["train"][0]
    print(f"[{mode:4s}] cortes: entrenamiento={len(ds['train'])}, validación={len(ds['val'])}, prueba={len(ds['test'])} | "
          f"x {x.shape} {x.dtype}, y {y.shape} valores {sorted(np.unique(y).tolist())}")
print("Cortes de entrenamiento con articulación (partición 0):", int(ds["train"].positive_flags().sum()))

## 11. Guardado y verificación

**Qué se hace.** Se escribe en `data/unet/`:
- `SIJ_<n>.npz` con `image` (z-score, 416×416, `float32`), `mask` (`uint8` 0/1) y `body` (máscara del cuerpo, útil para métricas o figuras).
- `splits.json` con las 9 particiones.
- `unet_prep_params.json` con todas las decisiones: tamaño, relleno, normalización y sus estadísticas por volumen, aumento, origen de las máscaras e inventario de cortes.

Después se relee todo con `load_unet_volumes` (la misma función que usará el entrenamiento) y se compara con lo que está en memoria.

**Por qué.** Igual que en el notebook 01: guardar sin releer es una fuente clásica de errores silenciosos. El archivo de parámetros permite saber, meses después o al escribir el artículo, exactamente con qué datos se entrenó cada modelo. En particular, **`mask_source` dice si las máscaras eran reales o provisionales**; el entrenamiento debería negarse a reportar resultados si son provisionales.

**Qué mirar.** "Contenido idéntico: True" en los 9 volúmenes.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in PATIENTS:
    z, m, b = prepared[p]
    np.savez_compressed(OUT_DIR / f"{p}.npz", image=z, mask=m, body=b)
(OUT_DIR / "splits.json").write_text(json.dumps(folds, indent=2), encoding="utf-8")

params = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source_dir": str(PROCESSED_DIR.relative_to(ROOT)),
    "mask_source": "provisional" if USING_PLACEHOLDER else "real",
    "mask_source_per_volume": mask_source,
    "target_size": TARGET_SIZE,
    "normalization": "z-score por volumen dentro de body_mask (low_threshold), sobre el uint8 del notebook 01",
    "padding": "constante con el valor del aire normalizado; centrado",
    "modes": {"2d": "1 canal (corte k)", "2.5d": "3 canales (k-1, k, k+1), borde repetido"},
    "splits": "leave-one-patient-out; validación = paciente siguiente (circular)",
    "augmentation": AugmentConfig().__dict__,
    "seed": SEED,
    "volumes": {p: {**norm_stats[p], **{k: v for k, v in inventory[p].items()}} for p in PATIENTS},
}
save_params_json(OUT_DIR / "unet_prep_params.json", params)

reloaded = load_unet_volumes(OUT_DIR)
assert list(reloaded) == PATIENTS
print(f"{'volumen':8s} {'imagen':>18s} {'máscara':>18s}  contenido idéntico")
for p in PATIENTS:
    zi, mi = reloaded[p]
    ok = np.array_equal(zi, prepared[p][0]) and np.array_equal(mi, prepared[p][1])
    print(f"{p:8s} {str(zi.shape) + ' ' + str(zi.dtype):>18s} {str(mi.shape) + ' ' + str(mi.dtype):>18s}  {ok}")
    assert ok
print("\nArchivos en", OUT_DIR.relative_to(ROOT), ":", sorted(f.name for f in OUT_DIR.iterdir()))

## 12. Resumen, decisiones abiertas y pasos siguientes

**Qué se produjo.** En `data/unet/` quedan los 9 volúmenes normalizados (z-score por volumen dentro del cuerpo), rellenados a 416×416 y emparejados con su máscara, junto con las 9 particiones por paciente y el archivo de parámetros. `SliceDataset` entrega los cortes en modo 2D o 2.5D, con aumento solo en entrenamiento.

**Decisiones tomadas y su motivo, en una línea.**
| Decisión | Motivo |
|---|---|
| z-score por volumen, dentro del cuerpo | Entrada centrada para la red, comparable entre pacientes y coherente entre cortes vecinos. |
| Relleno a 416 | Múltiplo de 32 para la U-Net, sin perder tejido ni resolución, y reversible. |
| 2.5D con ±1 corte y borde repetido | Contexto 3D barato, compatible con codificadores de 3 canales y sin canales artificiales. |
| LOPO con paciente de validación | Sin fuga entre pacientes, 9 mediciones para la media y la variabilidad, y prueba independiente. |
| Aumento moderado y solo plausible | Combate el sobreajuste sin enseñar anatomía imposible. |

**Decisiones abiertas (para el equipo).**
1. **Formato y criterio de las máscaras** (sección 3): una o dos clases, qué estructura exactamente es "la articulación" y si se anotan todos los cortes.
2. **LOPO (18 entrenamientos) o 3 particiones (6 entrenamientos)**, según el tiempo disponible en Colab.
3. **Sobremuestreo de cortes positivos**: decidirlo al ver el entrenamiento con máscaras reales.

**Pasos siguientes.**
1. Cuando lleguen las máscaras reales, copiarlas a `data/masks/` y volver a ejecutar este notebook. La tabla de la sección 3 debe decir "real" en los 9 volúmenes.
2. Notebook 03, de entrenamiento: U-Net con codificador preentrenado (por ejemplo, `segmentation_models_pytorch`), pérdida Dice + BCE, mismo protocolo para 2D y 2.5D, y Dice 3D por paciente en el conjunto de prueba de cada partición.
3. Para la app: `prepare_volume` → modelo → `unpad`, con las mismas funciones de `sij_prep.unet_data`.